# Running the OMOS Algorithm

In [ ]:
import sys
sys.path.append('../../')
import jax
import jax.numpy as jnp
import numpy as np
from jax_meta_rl.meta_rl.algorithms.omos import OMOS

# --- Hyperparameters ---
OBS_DIM = 10
ACTION_DIM = 3
META_BATCH_SIZE = 4
SUPPORT_SET_SIZE = 20
QUERY_SET_SIZE = 20
ONLINE_BATCH_SIZE = 32
NUM_META_UPDATES = 10
KEY = jax.random.PRNGKey(0)

# --- Dummy Data Generation ---
def generate_dummy_batch(key, batch_size, obs_dim, action_dim):
    obs_key, action_key, reward_key, next_obs_key, done_key = jax.random.split(key, 5)
    obs = jax.random.normal(obs_key, (batch_size, obs_dim))
    actions = jax.random.normal(action_key, (batch_size, action_dim))
    rewards = jax.random.uniform(reward_key, (batch_size,))
    next_obs = jax.random.normal(next_obs_key, (batch_size, obs_dim))
    dones = jax.random.randint(done_key, (batch_size,), 0, 2)
    return obs, actions, rewards, next_obs, dones

def generate_meta_batch(key, meta_batch_size, support_size, query_size, obs_dim, action_dim):
    tasks = []
    for i in range(meta_batch_size):
        key, support_key, query_key = jax.random.split(key, 3)
        support_batch = generate_dummy_batch(support_key, support_size, obs_dim, action_dim)
        query_batch = generate_dummy_batch(query_key, query_size, obs_dim, action_dim)
        tasks.append((support_batch, query_batch))
    return jax.tree_util.tree_map(lambda *xs: jnp.stack(xs), *tasks)

# --- Initialize Algorithm ---
omos_agent = OMOS(obs_dim=OBS_DIM, action_dim=ACTION_DIM)
KEY, init_key = jax.random.split(KEY)
params, opt_states = omos_agent.init_params(init_key)

# --- Meta-Training Loop ---
for i in range(NUM_META_UPDATES):
    # --- Offline Meta-RL Update ---
    KEY, meta_batch_key = jax.random.split(KEY)
    meta_batch = generate_meta_batch(meta_batch_key, META_BATCH_SIZE, SUPPORT_SET_SIZE, QUERY_SET_SIZE, OBS_DIM, ACTION_DIM)
    KEY, train_key = jax.random.split(KEY)
    params, opt_states['main'], total_loss = omos_agent.outer_update(params, opt_states['main'], meta_batch, train_key)

    # --- Online Self-Supervised Update ---
    KEY, online_key = jax.random.split(KEY)
    obs, actions, _, next_obs, _ = generate_dummy_batch(online_key, ONLINE_BATCH_SIZE, OBS_DIM, ACTION_DIM)
    online_batch_supervised = (obs, actions, next_obs)
    params, opt_states, dynamics_loss = omos_agent.self_supervised_update(params, opt_states, online_batch_supervised)
    
    if i % 2 == 0:
        print(f'Update {i}, Meta Loss: {total_loss:.4f}, Dynamics Loss: {dynamics_loss:.4f}')

print("Training loop complete.")